In [1]:
!pip install -q transformers datasets evaluate jiwer accelerate
!pip install -q yt-dlp pydub ffmpeg-python kagglehub
!apt-get install -q -y ffmpeg

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.


In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [3]:
import kagglehub

# Download latest version
dataset_path = kagglehub.dataset_download("mayarjao/arabic-tts")

print("Path to dataset files:", dataset_path)

Using Colab cache for faster access to the 'arabic-tts' dataset.
Path to dataset files: /kaggle/input/arabic-tts


In [4]:
meta_path = "/kaggle/input/arabic-tts/arabic_tts/metadata-wav.csv"
wavs_dir  = "/kaggle/input/arabic-tts/arabic_tts/wavs"

In [5]:
df = pd.read_csv(meta_path, sep='|', header=None, names=['text', 'file_id', 'text_norm'])
df['file_id']    = df['file_id'].str.strip().str.replace('wavs/', '', regex=False)
df['audio_path'] = df['file_id'].apply(lambda x: os.path.join(wavs_dir, x))
df               = df[df['audio_path'].apply(os.path.exists)].reset_index(drop=True)
df['sentence']   = df['text'].str.strip()
print(f"Valid samples: {len(df)}")

Valid samples: 78720


In [6]:
N_SAMPLES = 5000  # set to None to use all 78k
if N_SAMPLES:
    df = df.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")

Train: 4750 | Eval: 250


In [7]:
def df_to_hf(df):
    return Dataset.from_dict({
        'audio':    df['audio_path'].tolist(),
        'sentence': df['sentence'].tolist()
    }).cast_column('audio', Audio(sampling_rate=16000))

dataset = DatasetDict({
    'train': df_to_hf(train_df),
    'eval':  df_to_hf(eval_df)
})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 4750
    })
    eval: Dataset({
        features: ['audio', 'sentence'],
        num_rows: 250
    })
})


In [8]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Arabic", task="transcribe")
model     = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="Arabic", task="transcribe")
model.generation_config.suppress_tokens    = []

print("Whisper loaded:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Whisper loaded: openai/whisper-small


In [9]:
import torch

# ── Pre-process and cache everything once ─────────────────────────────────────
def preprocess(batch):
    audio = batch['audio']
    batch['input_features'] = processor.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

print("Pre-processing dataset (one time only)...")
dataset_processed = DatasetDict({
    'train': dataset['train'].map(preprocess, remove_columns=['audio', 'sentence']),
    'eval':  dataset['eval'].map(preprocess,  remove_columns=['audio', 'sentence'])
})

# Save to disk so you never redo this
dataset_processed.save_to_disk("/tmp/arabic_asr_processed")
print("Done! Saved to /tmp/arabic_asr_processed")

Pre-processing dataset (one time only)...


Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Saving the dataset (0/10 shards):   0%|          | 0/4750 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/250 [00:00<?, ? examples/s]

Done! Saved to /tmp/arabic_asr_processed


In [10]:
from datasets import load_from_disk

dataset_processed = load_from_disk("/tmp/arabic_asr_processed")
print(dataset_processed)

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 4750
    })
    eval: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 250
    })
})


In [11]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


In [12]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer (%)": round(wer, 2)}

print("WER metric ready.")

WER metric ready.


In [19]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./whisper-small-arabic",
    per_device_train_batch_size = 32,  # bigger = faster
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 1,   # effective batch = 32
    learning_rate               = 5e-4,
    warmup_steps                = 200,
    max_steps                   = 500,
    gradient_checkpointing      = False, # saves VRAM so we can use bigger batch
    fp16                        = True,
    eval_strategy               = "steps",
    eval_steps                  = 500,  # less frequent eval = faster
    save_strategy               = "steps",
    save_steps                  = 500,
    logging_steps               = 100,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer (%)",
    greater_is_better           = False,
    predict_with_generate       = True,
    generation_max_length       = 225,
    report_to                   = ["none"],
    push_to_hub                 = False,
    remove_unused_columns       = False,
)
trainer = Seq2SeqTrainer(
    model         = model,
    args          = training_args,
    train_dataset = dataset_processed['train'],
    eval_dataset  = dataset_processed['eval'],
    processing_class     = processor.feature_extractor,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

print("Trainer configured. Ready to train!")

Trainer configured. Ready to train!


In [20]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [21]:
trainer.train()

# Save final model & processor
trainer.save_model("./whisper-small-arabic-final")
processor.save_pretrained("./whisper-small-arabic-final")
print("Model saved to ./whisper-small-arabic-final")

Step,Training Loss,Validation Loss,Wer (%)
500,0.254435,0.584478,64.230000


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./whisper-small-arabic-final


In [22]:
results = trainer.evaluate()
print("\n=== Evaluation Results ===")
for k, v in results.items():
    print(f"  {k}: {v}")


=== Evaluation Results ===
  eval_loss: 0.5844776630401611
  eval_wer (%): 64.23
  eval_runtime: 97.6022
  eval_samples_per_second: 2.561
  eval_steps_per_second: 0.164
  epoch: 3.3557046979865772


In [23]:
import torch

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for i in range(3):
    sample = dataset_processed['eval'][i]
    input_features = torch.tensor(sample['input_features']).unsqueeze(0).to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    pred  = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    label = processor.tokenizer.decode(sample['labels'], skip_special_tokens=True)

    print(f"Sample {i+1}")
    print(f"  Reference : {label}")
    print(f"  Prediction: {pred}")
    print()

Sample 1
  Reference : والذين يقولون ربنا اصرف عنا عذاب جهنم إن عذابها كان غراما
  Prediction: والذين يقولون رب نصرف عنا عذاب جهنم إن عذابها كان ضراما

Sample 2
  Reference : حسن لنطقك بادئا ومجاوبا
  Prediction: حسن لنظرك بادئا ومجاوبا

Sample 3
  Reference : ما رأيت أحدا أطيب عيشا منه ، مع ما كان فيه من ضيق العيش.
  Prediction: ما رأيت أحداً أطيباً منه معما كان فيه من بيق العيش.

